# Lekcija 28: Epipolarna geometrija

Stereo podudaranje iz lekcije 22 pretpostavljalo je *rektificirani* par, u kojem odgovarajuće točke uvijek leže u istom retku. U općenitijem slučaju stereo kamere ne moraju biti rektificirane. Ova lekcija obrađuje opći odnos između dvaju pogleda za **bilo koji** par slika statičnog prizora. Kao što ćemo vidjeti, odgovarajuća točka na drugoj slici nije samo *negdje* &mdash; mora ležati na određenom pravcu. Obrađujemo i kalibrirani i nekalibrirani slučaj.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Epipolarno ograničenje

Za točku $x_1$ na slici 1 i njezinu stvarnu odgovarajuću točku $x_2$ na slici 2 (obje u homogenim pikselnim koordinatama) postoji matrica $F$ dimenzija $3\times3$ nepotpunog ranga &mdash; **fundamentalna matrica** &mdash; takva da vrijedi

$$x_2^\top F x_1 = 0$$

za *svaki* par odgovarajućih točaka, neovisno o geometriji prizora. $F$ ovisi samo o relativnom položaju i orijentaciji dviju kamera i njihovim (nekalibriranim) unutarnjim parametrima. Nakon preuređenja, $l_2 = F x_1$ je **epipolarni pravac** na slici 2: jednodimenzijski pravac na kojem sigurno leži odgovarajuća točka za $x_1$. To je upravo mehanizam koji opravdava pretraživanje ograničeno na redak iz lekcije 22 &mdash; rektifikacija je samo poseban raspored kamera u kojem su svi epipolarni pravci vodoravni, pa se jednodimenzijsko pretraživanje svodi na jedan redak.

### Zašto pravac? Geometrijski prikaz

Kamera 1 zna samo *smjer* prema točki, a ne njezinu dubinu: stvarna 3D točka $X$ može biti bilo gdje na zraci iz $C_1$ kroz $x_1$. No svaka točka na toj zraci, zajedno s $C_1$ i $C_2$, leži u jednoj ravnini &mdash; **epipolarnoj ravnini**. Kamera 2 vidi tu ravninu s boka, kao jedan pravac: gdje god se $X$ stvarno nalazi na zraci, njezina projekcija na sliku 2 uvijek pada na taj pravac, epipolarni pravac.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

C1 = np.array([0.0, 0.0, 0.0])
C2 = np.array([3.0, 0.5, -0.5])
ray_dir = np.array([0.6, 0.8, 3.0])
ray_dir = ray_dir / np.linalg.norm(ray_dir)
depths = [2.0, 3.5, 5.0]                        # a few candidate depths along camera 1's ray
candidates = [C1 + d * ray_dir for d in depths]

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')

ax.plot(*zip(C1, C2), c='gray', linestyle='--', linewidth=1)
for X in candidates:
    ax.plot(*zip(C1, X), c='tab:blue', linewidth=1)
    ax.plot(*zip(C2, X), c='tab:green', linewidth=1)
cand_arr = np.array(candidates)
ax.scatter(cand_arr[:, 0], cand_arr[:, 1], cand_arr[:, 2], c='black', s=30)
ax.text(*candidates[-1], '  possible $X$\n  (unknown depth)', fontsize=8)

ax.scatter(*C1, c='tab:blue', s=60)
ax.text(*C1, '  $C_1$', fontsize=10)
ax.scatter(*C2, c='tab:green', s=60)
ax.text(*C2, '  $C_2$', fontsize=10)

plane = Poly3DCollection([[C1, C2, candidates[-1]]], alpha=0.15, facecolor='tab:orange')
ax.add_collection3d(plane)
centroid = (C1 + C2 + candidates[1]) / 3
ax.text(*centroid, 'epipolar\nplane', fontsize=8, color='tab:orange', ha='center')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('Why the match is constrained to a line')
plt.tight_layout()
plt.show()

Tri plava segmenta zrake su iz $C_1$ kroz isti piksel $x_1$, samo produljene do triju različitih, jednako mogućih dubina &mdash; kamera 1 sama ih ne može razlikovati. No svaki kandidat, zajedno s $C_1$ i $C_2$, i dalje leži u prikazanoj epipolarnoj ravnini, pa njihove projekcije u kameru 2 (krajevi zelenih segmenata) leže duž presjeka te ravnine sa slikom 2: epipolarnog pravca. $F$ je samo algebarski postupak koji predviđa taj pravac izravno iz $x_1$, bez poznavanja dubine.

## Sintetički prizor s dvjema kamerama

Izgradimo dvije kamere s poznatim unutarnjim parametrima $K$ i poznatom relativnom rotacijom i translacijom, stvorimo nekoliko nasumičnih 3D točaka, projiciramo ih u obje kamere i pomoću dobivenih korespondencija odredimo $F$.

In [ ]:
rng = np.random.default_rng(0)
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

R1, t1 = np.eye(3), np.zeros(3)                       # camera 1: at the origin, looking down +z
angle = np.radians(15)
R_true = np.array([[np.cos(angle), 0, np.sin(angle)],
                    [0, 1, 0],
                    [-np.sin(angle), 0, np.cos(angle)]])
t_true = np.array([0.5, 0.0, 0.1])                    # camera 2: rotated 15 deg, shifted along x

P1 = K @ np.hstack([R1, t1.reshape(3, 1)])
P2 = K @ np.hstack([R_true, t_true.reshape(3, 1)])

def project(P, points_3d):
    homogeneous = np.hstack([points_3d, np.ones((len(points_3d), 1))])
    projected = (P @ homogeneous.T).T
    return projected[:, :2] / projected[:, 2:3]

points_3d = rng.uniform(-1, 1, (40, 3)) + np.array([0, 0, 5])  # in front of both cameras
x1 = project(P1, points_3d)
x2 = project(P2, points_3d)

print(f'{len(points_3d)} 3D points, projected into both cameras')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].scatter(x1[:, 0], x1[:, 1], c='tab:blue', s=15)
axes[0].set_xlim(0, 640); axes[0].set_ylim(480, 0)
axes[0].set_title('Image 1')
axes[1].scatter(x2[:, 0], x2[:, 1], c='tab:orange', s=15)
axes[1].set_xlim(0, 640); axes[1].set_ylim(480, 0)
axes[1].set_title('Image 2')
plt.tight_layout()
plt.show()

### SVD: dekompozicija na singularne vrijednosti

U lekciji 23 vidjeli smo da se realna simetrična pozitivno semidefinitna matrica $A$ (poput matrice kovarijance) može zapisati kao $A = P \Lambda P^\top$, pri čemu $P$ sadrži *svojstvene vektore*, a dijagonalna i nenegativna $\Lambda$ sadrži *svojstvene vrijednosti*. **Dekompozicija na singularne vrijednosti (SVD)** poopćuje tu ideju: bilo koja matrica $A$ &mdash; simetrična ili ne, kvadratna ili ne &mdash; može se zapisati kao $A = U \Sigma V^\top$, pri čemu su $U$ i $V$ ortogonalne i sadrže **singularne vektore** (lijeve odnosno desne), a dijagonalna i nenegativna $\Sigma$ sadrži **singularne vrijednosti**. Po definiciji su singularne vrijednosti u $\Sigma$ poredane od najveće prema najmanjoj. Za naše potrebe usredotočit ćemo se na dvije činjenice o SVD-u:

**Rješavanje $Ax=0$ što je moguće točnije, uz uvjet $\|x\|=1$.** Rješenje je posljednji stupac matrice $V$ (odnosno posljednji redak matrice $V^\top$) &mdash; desni singularni vektor pridružen najmanjoj singularnoj vrijednosti. To je poopćenje postupka potpunih najmanjih kvadrata iz lekcije 23: stupci matrice $V$ svojstveni su vektori matrice $A^\top A$, poredani prema svojstvenoj vrijednosti. Stoga je smjer najmanje singularne vrijednosti ovdje upravo isti „smjer najmanje pogreške prilagodbe” kao svojstveni vektor najmanje svojstvene vrijednosti iz lekcije 23.

**Nametanje ograničenja ranga.** Postavljanje najmanjih singularnih vrijednosti na nulu i rekonstrukcija daju matricu nižeg ranga najbližu izvorniku (u smislu najmanjih kvadrata). Upravo je to potrebno da se procijenjeni $F$ &mdash; koji mora imati točno rang 2 &mdash; vrati na to ograničenje nakon što neograničeno rješavanje s 9 parametara neizbježno malo odstupi od njega.

## Algoritam s 8 točaka

Svaka korespondencija daje jednu linearnu jednadžbu s 9 nepoznatih elemenata matrice $F$ (razvojem izraza $x_2^\top F x_1 = 0$). $F$ je određena samo do faktora mjerila (poput homografije iz lekcije 24), pa zapravo ima samo 8 neovisnih nepoznanica, koje se mogu odrediti kao homogeni problem najmanjih kvadrata pomoću SVD-a (gore) &mdash; istim postupkom DLT-a koji se koristio za prilagodbu homografije u lekciji 24. Budući da $F$ mora imati točno rang 2 (fundamentalna matrica uvijek je singularna), rješenje s 9 parametara projiciramo natrag na najbližu matricu ranga 2 pomoću prethodnog postupka sa SVD-om.

(Primijetite da $F$ zapravo ima samo 7 stvarnih stupnjeva slobode, jer ograničenje ranga 2 uklanja još jedan. Međutim, svaki algoritam sa 7 točaka bit će složeniji jer mora uzeti u obzir nelinearno ograničenje; algoritam s 8 točaka praktičan je jer je *linearan*, iako zahtijeva zaseban korak za ponovno uspostavljanje ranga 2.)

Postoji još jedna poteškoća prije nego što problem predamo SVD-u: sirove pikselne koordinate (reda nekoliko stotina) čine linearni sustav numerički loše uvjetovanim. Zato najprije skaliramo točke svake slike tako da im je središte u ishodištu, a prosječna udaljenost od njega $\sqrt2$, rješavamo u tim normaliziranim koordinatama, a zatim poništavamo skaliranje dobivene matrice $F$. Taj korak poboljšanja uvjetovanosti, koji u praksi donosi veliku razliku, pretvara obični algoritam s 8 točaka u **normalizirani algoritam s 8 točaka** (<a href="../references.html#hartley-1997">Hartley, 1997.</a>) &mdash; varijantu koju koristimo u nastavku i koju je dobro koristiti u praksi.

In [ ]:
def normalize_points(x):
    """Shift/scale so points are centered at the origin with average distance sqrt(2) -- standard
    numerical-conditioning trick (Hartley normalization) for the 8-point algorithm."""
    mean = x.mean(axis=0)
    std = x.std()
    T = np.array([[1 / std, 0, -mean[0] / std],
                  [0, 1 / std, -mean[1] / std],
                  [0, 0, 1]])
    x_h = np.hstack([x, np.ones((len(x), 1))])
    return (T @ x_h.T).T, T

def eight_point_algorithm(x1, x2):
    x1n, T1 = normalize_points(x1)
    x2n, T2 = normalize_points(x2)

    A = np.array([[xb * xa, xb * ya, xb, yb * xa, yb * ya, yb, xa, ya, 1]
                  for (xa, ya, _), (xb, yb, _) in zip(x1n, x2n)])
    _, _, Vt = np.linalg.svd(A)
    F = Vt[-1].reshape(3, 3)

    U, S, Vt2 = np.linalg.svd(F)   # enforce rank-2 by zeroing the smallest singular value
    S[-1] = 0
    F = U @ np.diag(S) @ Vt2

    F = T2.T @ F @ T1              # undo the normalization
    return F / F[2, 2]

F_mine = eight_point_algorithm(x1, x2)
F_cv, _ = cv2.findFundamentalMat(x1, x2, cv2.FM_8POINT)

print('our F:\n', np.round(F_mine, 5))
print('cv2.findFundamentalMat F:\n', np.round(F_cv, 5))

### Izravna provjera epipolarnog ograničenja

Za stvarne korespondencije $x_2^\top F x_1$ bi trebalo biti (numerički) jednako nuli.

In [ ]:
def epipolar_residual(F, x1, x2):
    x1h = np.hstack([x1, np.ones((len(x1), 1))])
    x2h = np.hstack([x2, np.ones((len(x2), 1))])
    return np.abs(np.sum(x2h * (F @ x1h.T).T, axis=1))

print(f'mean |x2^T F x1|, our F:  {epipolar_residual(F_mine, x1, x2).mean():.2e}')
print(f'mean |x2^T F x1|, cv2 F:  {epipolar_residual(F_cv, x1, x2).mean():.2e}')

### Vizualizacija epipolarnih pravaca

Za nekoliko točaka na slici 1 crtamo njihove epipolarne pravce $l_2 = Fx_1$ na slici 2 i potvrđujemo da svaka stvarna odgovarajuća točka leži točno na svom pravcu.

In [ ]:
w, h = 640, 480
sample_idx = rng.choice(len(x1), 6, replace=False)
colors = plt.cm.tab10.colors[:len(sample_idx)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(x1[:, 0], x1[:, 1], c='gray', s=15)
axes[0].scatter(x1[sample_idx, 0], x1[sample_idx, 1], c=colors, s=40)
axes[0].set_xlim(0, w); axes[0].set_ylim(h, 0)
axes[0].set_title('Image 1: 6 selected points')

axes[1].scatter(x2[:, 0], x2[:, 1], c='gray', s=15)
for i, color in zip(sample_idx, colors):
    a, b, c = F_mine @ np.array([x1[i, 0], x1[i, 1], 1])   # line: a*x + b*y + c = 0
    xs = np.array([0, w])
    ys = -(a * xs + c) / b
    axes[1].plot(xs, ys, linewidth=1, color=color)
axes[1].scatter(x2[sample_idx, 0], x2[sample_idx, 1], c=colors, s=40, zorder=5,
                edgecolors='black', linewidths=0.6, label='true match')
axes[1].set_xlim(0, w); axes[1].set_ylim(h, 0)
axes[1].set_title('Image 2: epipolar lines + true matches')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

Svaka stvarna odgovarajuća točka pada točno na predviđeni pravac &mdash; pretraživanje točke na drugoj slici doista se svodi s 2D na 1D, čak i bez prethodne rektifikacije slika.

## Od fundamentalne do esencijalne matrice: dodavanje kalibracije

Fundamentalna matrica radi u sirovim pikselnim koordinatama, uz pretpostavku da unutarnji parametri kamere nisu poznati. Ako *znamo* unutarnje parametre $K$ (iz kalibracije kamere &mdash; lekcija 27), možemo ukloniti njihov utjecaj i raditi u normaliziranim koordinatama, čime dobivamo **esencijalnu matricu**:

$$E = K_2^\top F K_1 \qquad \text{(if both are the same camera, then } K_1 = K_2 = K \text{)}$$

Za razliku od $F$, koja ima 7 stupnjeva slobode, $E$ ima samo 5 &mdash; u potpunosti je izgrađena iz relativne rotacije $R$ i smjera translacije $t$ između dviju kamera: $E = [t]_\times R$, gdje je $[t]_\times$ antisimetrična „matrica vektorskog umnoška” dimenzija $3\times3$ za $t$, definirana tako da vrijedi $[t]_\times v = t \times v$ za bilo koji vektor $v$. To znači da se $E$ može ponovno *rastaviti* na $R$ i $t$, čime se gibanje kamere određuje samo iz korespondencija na slikama.

Robusno procjenjujemo $E$ pomoću `cv2.findEssentialMat(..., method=cv2.RANSAC, ...)`, upravo onom vrstom prilagodbe koja odbacuje izdvojene vrijednosti, a koju smo izgradili od početka u lekciji 23.

In [ ]:
E_from_F = K.T @ F_mine @ K
E_direct, _ = cv2.findEssentialMat(x1, x2, K, method=cv2.RANSAC, threshold=1.0)

# E is also only defined up to scale -- compare directions, not raw magnitudes
print('E derived from our F (normalized):\n', np.round(E_from_F / np.linalg.norm(E_from_F), 4))
print('E from cv2.findEssentialMat (normalized):\n', np.round(E_direct / np.linalg.norm(E_direct), 4))

### Određivanje gibanja kamere

`cv2.recoverPose` rastavlja $E$ na rotaciju i *smjer* translacije &mdash; ne na potpuni vektor translacije, jer se njezina veličina načelno ne može odrediti samo iz dvaju pogleda. Dvostruko veći prizor, promatran kamerama s dvostrukim međusobnim razmakom, daje identične slike; to je ista neodređenost mjerila poznata iz monokularnog vida.

In [ ]:
_, R_estimated, t_estimated, _ = cv2.recoverPose(E_direct, x1, x2, K)

print('true rotation:\n', np.round(R_true, 4))
print('recovered rotation:\n', np.round(R_estimated, 4))
print()
print('true translation direction:      ', np.round(t_true / np.linalg.norm(t_true), 4))
print('recovered translation direction: ', np.round(t_estimated.ravel(), 4))

Rotacija je određena, a smjer translacije podudara se do očekivanog predznaka i mjerila. Samo iz korespondencija značajki (lekcija 20) odredili smo relativnu rotaciju i translaciju (do faktora mjerila) između dviju kamera. To je polazište za strukturu iz gibanja (lekcija 29).

### Zadaci

1. Dodajte pikselni šum (npr. standardne devijacije 0.5) u `x1` i `x2` prije pokretanja algoritma s 8 točaka. Koliko raste prosječni epipolarni rezidual i je li normalizacija koordinata (koju provodi `eight_point_algorithm`) ovdje stvarno važna &mdash; pokušajte preskočiti korak normalizacije i usporedite.
2. Epipol na slici 2 projekcija je središta kamere 1, a vrijedi $F e_1 = 0$ za epipol $e_1$ na slici 1 (i $F^\top e_2 = 0$ za epipol $e_2$ na slici 2). Izračunajte oba epipola kao nul-prostor matrice $F$ (pomoću SVD-a) i provjerite nalaze li se unutar ili izvan vidljivog područja slike za ovaj raspored kamera.
3. Povećajte kut rotacije između dviju kamera na 60 stupnjeva i ponovno odredite njihov položaj i orijentaciju. Pronalazi li `cv2.recoverPose` i dalje točnu rotaciju? Kada biste očekivali da samo podudaranje korespondencija (lekcija 20) postane usko grlo umjesto geometrije?